In [12]:
import torch
import numpy as np


class WeaklyConvexProblem:
    """
    Defines the objective function phi(x) = f(x) + r(x).
    Based on Example 2.1 (Phase Retrieval) or simple |x^2 - 1|.
    """

    def __init__(self, objective_fn, rho = 2.0):
        self.phi = objective_fn
        self.rho = rho  # Weak convexity parameter

    def objective(self, x):
        """The true objective phi(x) = |x^2 - 1|"""
        return self.phi(x)

    def stochastic_model(self, x_t):
        """
        Returns a 'model' function g_xt(y) centered at x_t.
        This represents the stochastic one-sided model[cite: 49, 382].
        """
        # Linearization at x_t (Stochastic Subgradient Model)
        x_t_temp = x_t.detach().requires_grad_(True)
        f_val = self.objective(x_t_temp)
        f_val.backward()
        
        f_val_const = f_val.detach()
        grad_const = x_t_temp.grad.detach()

        def model(y):
            # f(x_t) + <grad, y - x_t>
            return f_val_const + torch.dot(grad_const.flatten(), (y - x_t).flatten())
        
        return model

In [13]:
class ModelBasedSolver:
    """
    Implements Algorithm 1.2: Stochastic Model-Based Minimization .
    """
    def __init__(self, problem, x_init, T, gamma=0.1):
        self.prob = problem
        self.x = x_init.clone().detach()
        self.T = T
        self.gamma = gamma # Step-size parameter
        self.history = []

    def run(self):
        # Step size sequence alpha_t = gamma / sqrt(T+1)
        alpha_t = self.gamma / np.sqrt(self.T + 1)
        
        for t in range(self.T):
            x_t = self.x.clone().detach()
            g_xt = self.prob.stochastic_model(x_t)
            
            # SUBPROBLEM: x_{t+1} = argmin { f_xt(y) + (1/2*alpha) * ||y-x_t||^2 }
            y = x_t.clone().requires_grad_(True)
            inner_opt = torch.optim.LBFGS([y], lr=1, max_iter=20)

            def closure():                
                inner_opt.zero_grad()
                model_val = g_xt(y)
                penalty = (1.0 / (2 * alpha_t)) * torch.norm(y - x_t)**2
                loss = model_val + penalty
                loss.backward()
                return loss

            inner_opt.step(closure)
            self.x = y.detach().clone()
            
            if t % 50 == 0:
                print(f"Iter {t}: x = {self.x.item():.4f}, f(x) = {self.prob.objective(self.x).item():.4f}")

        return self.x

In [6]:
def abs_parabola_phi(x):
    return torch.abs(x**2 - 1.0)

In [14]:
# --- Main Execution ---
T_total = 500
prob = WeaklyConvexProblem(abs_parabola_phi, rho=2.0)
# lambda < 1/rho (e.g., 1/2.1)
solver = ModelBasedSolver(prob, x_init=torch.tensor([2.0]), T=T_total, gamma=0.2)

final_x = solver.run()
print(f"\nOptimization Finished. Final x: {final_x.item():.4f}")

Iter 0: x = 1.9643, f(x) = 2.8583
Iter 50: x = 0.9880, f(x) = 0.0238
Iter 100: x = 1.0159, f(x) = 0.0320
Iter 150: x = 1.0078, f(x) = 0.0156
Iter 200: x = 0.9998, f(x) = 0.0005
Iter 250: x = 0.9918, f(x) = 0.0163
Iter 300: x = 0.9839, f(x) = 0.0319
Iter 350: x = 1.0116, f(x) = 0.0234
Iter 400: x = 1.0036, f(x) = 0.0071
Iter 450: x = 0.9956, f(x) = 0.0088

Optimization Finished. Final x: 1.0056


In [17]:
def max_parabola_phi(x):
    """The 'True' Objective: max(x^2, (x-4)^2)"""
    f1 = x**2
    f2 = (x - 4.0)**2
    return torch.max(f1, f2)

def max_parabola_model_gen(x_t):
    """
    Returns the Stochastic Model for the Pointwise Max.
    Based on Example 2.6 and Section 4.2[cite: 984].
    """
    # Detach constants for the inner loop
    x_t_val = x_t.detach()
    f1_val = x_t_val**2
    f2_val = (x_t_val - 4.0)**2
    
    grad1 = 2 * x_t_val
    grad2 = 2 * (x_t_val - 4.0)

    def model(y):
        # Linearize both components inside the max
        l1 = f1_val + grad1 * (y - x_t_val)
        l2 = f2_val + grad2 * (y - x_t_val)
        return torch.max(l1, l2)
    
    return model

# --- Setup and Run ---
prob_max = WeaklyConvexProblem(max_parabola_phi, rho=2.0)
# Overriding the model generator specifically for this non-linear model
prob_max.stochastic_model = max_parabola_model_gen 

solver_max = ModelBasedSolver(prob_max, x_init=torch.tensor([10.0]), T=500, gamma=0.5)
final_x_max = solver_max.run()

print(f"Final x for Pointwise Max: {final_x_max.item():.4f}")

Iter 0: x = 9.5532, f(x) = 91.2643
Iter 50: x = 1.9946, f(x) = 4.0217
Iter 100: x = 1.9565, f(x) = 4.1760
Iter 150: x = 2.0264, f(x) = 4.1061
Iter 200: x = 1.9945, f(x) = 4.0221
Iter 250: x = 1.9565, f(x) = 4.1761
Iter 300: x = 2.0264, f(x) = 4.1061
Iter 350: x = 1.9945, f(x) = 4.0221
Iter 400: x = 1.9565, f(x) = 4.1761
Iter 450: x = 2.0264, f(x) = 4.1061
Final x for Pointwise Max: 2.0435
